# 📝 Async Programming (asyncio, aiohttp, async generators)
### Exercises & Solutions — 25 Problems

This notebook is exercises-and-solutions only. It assumes you've already covered the
concept notebook for this topic. Each problem targets a **distinct function, pattern,
or real-world scenario** so that working through all of them gives you practical
exposure to everything commonly used on the job.

**Coverage map:**

- Coroutine basics, await, sequential vs concurrent (1-5)
- Tasks: create_task, gather, wait, as_completed (6-11)
- Timeouts & cancellation (12-14)
- Synchronization: Lock, Semaphore, Event, Queue (15-19)
- Async generators & comprehensions (20-22)
- Real patterns: rate limiting, fan-out/fan-in, async context managers (23-25)


---


### 1. Your First Coroutine and Awaiting It

Write a coroutine `greet(name, delay)` that sleeps then returns a greeting, and await it directly.

In [ ]:
import asyncio

async def greet(name, delay):
    await asyncio.sleep(delay)
    return f"Hello, {name}!"

result = await greet("World", 0.1)
print(result)

### 2. Calling a Coroutine Without Awaiting (the classic mistake)

Demonstrate what happens if you call a coroutine WITHOUT awaiting it (returns a coroutine object, does nothing), then fix it.

In [ ]:
import asyncio

async def compute():
    await asyncio.sleep(0.01)
    return 42

result_wrong = compute()      # NOT awaited!
print(f"Without await: {result_wrong} (type: {type(result_wrong).__name__}) - did nothing!")

result_right = await compute()
print(f"With await: {result_right}")

# Clean up the never-awaited coroutine to avoid a warning
result_wrong.close()

### 3. Sequential Execution Timing Baseline

Time 3 sequential `await asyncio.sleep()` calls and confirm total time ≈ sum of delays.

In [ ]:
import asyncio, time

async def task(name, delay):
    await asyncio.sleep(delay)
    return name

start = time.perf_counter()
r1 = await task("A", 0.1)
r2 = await task("B", 0.1)
r3 = await task("C", 0.1)
elapsed = time.perf_counter() - start
print(f"Sequential: {elapsed:.2f}s for 3x 0.1s tasks (expect ~0.3s)")

### 4. asyncio.gather for Concurrent Execution

Run the SAME 3 tasks concurrently with `asyncio.gather` and confirm total time ≈ max(delays), not sum.

In [ ]:
import asyncio, time

async def task(name, delay):
    await asyncio.sleep(delay)
    return name

start = time.perf_counter()
results = await asyncio.gather(task("A", 0.1), task("B", 0.1), task("C", 0.1))
elapsed = time.perf_counter() - start
print(f"Concurrent: {elapsed:.2f}s for 3x 0.1s tasks (expect ~0.1s)")
print("Results (order preserved, matching input order):", results)

### 5. gather Preserves Order Even With Different Delays

Run tasks with delays [0.3, 0.1, 0.2] via gather and confirm results come back in ORIGINAL submission order, not completion order.

In [ ]:
import asyncio

async def task(name, delay):
    await asyncio.sleep(delay)
    return f"{name} (took {delay}s)"

results = await asyncio.gather(task("first", 0.3), task("second", 0.1), task("third", 0.2))
print(results)
print("Note: 'second' finishes FIRST internally, but gather still returns results in submission order")

### 6. create_task for Fire-and-Continue

Use `asyncio.create_task` to start a task, do OTHER work while it runs in the background, then await its result.

In [ ]:
import asyncio

async def background_job():
    await asyncio.sleep(0.2)
    return "background result"

async def main():
    task = asyncio.create_task(background_job())   # starts running immediately
    print("Task launched, doing other synchronous-ish work...")
    await asyncio.sleep(0.05)
    print("Still waiting for background task...")
    result = await task                              # now actually wait for it
    return result

print(await main())

### 7. Tracking Multiple Tasks in a List

Create 5 tasks in a loop, store them in a list, then gather all results — demonstrating the common 'fan-out' pattern.

In [ ]:
import asyncio

async def fetch(item_id):
    await asyncio.sleep(0.05)
    return f"item-{item_id}-data"

tasks = [asyncio.create_task(fetch(i)) for i in range(5)]
results = await asyncio.gather(*tasks)
print(results)

### 8. asyncio.wait for Fine-Grained Control

Use `asyncio.wait` (instead of gather) with `return_when=FIRST_COMPLETED` to react as soon as ANY task finishes, leaving others still running.

In [ ]:
import asyncio

async def task(name, delay):
    await asyncio.sleep(delay)
    return name

tasks = [asyncio.create_task(task(n, d)) for n, d in [("slow", 0.3), ("fast", 0.05), ("medium", 0.15)]]
done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)

print(f"Completed first: {[t.result() for t in done]}")
print(f"Still pending: {len(pending)} task(s)")

# Clean up remaining tasks
for t in pending:
    await t
print("All tasks eventually finished")

### 9. as_completed for Processing Results as They Arrive

Use `asyncio.as_completed` to process each task's result THE MOMENT it finishes, in completion order (not submission order).

In [ ]:
import asyncio

async def task(name, delay):
    await asyncio.sleep(delay)
    return name

coros = [task("slow", 0.3), task("fast", 0.05), task("medium", 0.15)]
print("Processing in COMPLETION order (fast should print first):")
for coro in asyncio.as_completed(coros):
    result = await coro
    print(f"  Completed: {result}")

### 10. Handling Exceptions in gather (return_exceptions)

Run a mix of succeeding and failing coroutines via `gather(..., return_exceptions=True)` so ONE failure doesn't cancel the others.

In [ ]:
import asyncio

async def risky(n):
    await asyncio.sleep(0.02)
    if n == 2:
        raise ValueError(f"task {n} failed")
    return f"task {n} ok"

results = await asyncio.gather(*[risky(i) for i in range(4)], return_exceptions=True)
for r in results:
    if isinstance(r, Exception):
        print(f"  Error: {r}")
    else:
        print(f"  Success: {r}")

### 11. Without return_exceptions — One Failure Cancels Nothing But Raises

Show that WITHOUT `return_exceptions=True`, `gather` raises the first exception immediately to the caller (though other tasks still run to completion in background).

In [ ]:
import asyncio

async def risky(n):
    await asyncio.sleep(0.02)
    if n == 1:
        raise ValueError(f"task {n} failed")
    return f"task {n} ok"

try:
    results = await asyncio.gather(*[risky(i) for i in range(4)])
except ValueError as e:
    print(f"gather raised immediately: {e}")
    print("(other tasks may still complete in the background, but we don't see their results here)")

### 12. wait_for with Timeout — Success Case

Use `asyncio.wait_for` on a task that completes WITHIN the timeout, confirming normal results pass through.

In [ ]:
import asyncio

async def quick_task():
    await asyncio.sleep(0.05)
    return "done in time"

result = await asyncio.wait_for(quick_task(), timeout=0.5)
print(result)

### 13. wait_for with Timeout — Failure Case

Use `asyncio.wait_for` on a task that EXCEEDS the timeout, catching `asyncio.TimeoutError` and providing a fallback.

In [ ]:
import asyncio

async def slow_task():
    await asyncio.sleep(2)
    return "should never get here"

try:
    result = await asyncio.wait_for(slow_task(), timeout=0.1)
except asyncio.TimeoutError:
    result = "fallback: operation timed out"
print(result)

### 14. Manual Task Cancellation and Cleanup

Create a long-running task, cancel it manually mid-flight, and properly handle the resulting `CancelledError` with cleanup logic.

In [ ]:
import asyncio

async def long_running():
    try:
        print("  Starting long operation...")
        await asyncio.sleep(10)
        return "completed"
    except asyncio.CancelledError:
        print("  Caught cancellation, cleaning up...")
        raise   # re-raise so the task is properly marked cancelled

task = asyncio.create_task(long_running())
await asyncio.sleep(0.1)
task.cancel()
try:
    await task
except asyncio.CancelledError:
    print("Task cancellation confirmed by caller")
print("Task cancelled:", task.cancelled())

### 15. asyncio.Lock for Mutual Exclusion

Use an `asyncio.Lock` to protect a shared counter from concurrent coroutines incrementing it unsafely (race-condition-style, even though asyncio is single-threaded, ordering still matters across awaits).

In [ ]:
import asyncio

counter = {"value": 0}
lock = asyncio.Lock()

async def increment_unsafe(times):
    for _ in range(times):
        current = counter["value"]
        await asyncio.sleep(0)   # yield control - simulates a real async gap
        counter["value"] = current + 1

async def increment_safe(times):
    for _ in range(times):
        async with lock:
            current = counter["value"]
            await asyncio.sleep(0)
            counter["value"] = current + 1

counter["value"] = 0
await asyncio.gather(*[increment_unsafe(50) for _ in range(4)])
print(f"Unsafe result (expected 200): {counter['value']}")

counter["value"] = 0
await asyncio.gather(*[increment_safe(50) for _ in range(4)])
print(f"Safe result (expected 200): {counter['value']}")

### 16. asyncio.Semaphore for Bounded Concurrency

Use a `Semaphore(2)` to ensure only 2 'API calls' run concurrently out of 6 total, tracking the live concurrent count.

In [ ]:
import asyncio

sem = asyncio.Semaphore(2)
active = []
max_concurrent = [0]

async def api_call(n):
    async with sem:
        active.append(n)
        max_concurrent[0] = max(max_concurrent[0], len(active))
        await asyncio.sleep(0.05)
        active.remove(n)
    return n

results = await asyncio.gather(*[api_call(i) for i in range(6)])
print(f"Results: {results}")
print(f"Max concurrent observed: {max_concurrent[0]} (should never exceed 2)")

### 17. asyncio.Event for Signaling Between Coroutines

Use `asyncio.Event` to make a 'waiter' coroutine block until a 'setter' coroutine signals readiness.

In [ ]:
import asyncio

ready = asyncio.Event()

async def waiter(name):
    print(f"{name}: waiting for signal...")
    await ready.wait()
    print(f"{name}: signal received, proceeding!")

async def setter():
    await asyncio.sleep(0.1)
    print("Setter: sending signal now")
    ready.set()

await asyncio.gather(waiter("Worker-1"), waiter("Worker-2"), setter())

### 18. asyncio.Queue Producer-Consumer with join()

Build a producer-consumer pipeline using `asyncio.Queue`, using `queue.join()` to wait until ALL items are processed (not just produced).

In [ ]:
import asyncio

async def producer(queue, n):
    for i in range(n):
        await queue.put(f"item-{i}")
    print("Producer: done putting items")

async def consumer(queue, cid):
    while True:
        item = await queue.get()
        await asyncio.sleep(0.02)         # simulate processing
        print(f"  consumer-{cid} processed {item}")
        queue.task_done()

async def main():
    queue = asyncio.Queue()
    producer_task = asyncio.create_task(producer(queue, 6))
    consumer_tasks = [asyncio.create_task(consumer(queue, i)) for i in range(2)]

    await producer_task
    await queue.join()                    # waits until task_done() called for EVERY item
    print("All items processed!")

    for t in consumer_tasks:
        t.cancel()                         # consumers loop forever, must cancel explicitly

await main()

### 19. Bounded Queue Applying Natural Backpressure

Use `asyncio.Queue(maxsize=2)` to show that a fast producer is automatically slowed down (`put()` blocks) when consumers can't keep up.

In [ ]:
import asyncio, time

async def producer(queue, n):
    for i in range(n):
        start = time.perf_counter()
        await queue.put(i)
        wait_time = time.perf_counter() - start
        flag = " <- BLOCKED (queue was full)" if wait_time > 0.01 else ""
        print(f"  produced {i} (waited {wait_time:.3f}s){flag}")
    await queue.put(None)

async def consumer(queue):
    while True:
        item = await queue.get()
        if item is None:
            break
        await asyncio.sleep(0.05)   # slow consumer

queue = asyncio.Queue(maxsize=2)    # small buffer -> producer will block once full
await asyncio.gather(producer(queue, 6), consumer(queue))

### 20. Async Generator Basics

Write an async generator `countdown(n)` yielding values with awaits between, and consume it with `async for`.

In [ ]:
import asyncio

async def countdown(n):
    while n > 0:
        yield n
        await asyncio.sleep(0.02)
        n -= 1

async for value in countdown(5):
    print(value, end=" ")
print()

### 21. Async Comprehension with Filtering

Use an async list comprehension to collect only EVEN values from an async generator, in one expression.

In [ ]:
import asyncio

async def numbers(n):
    for i in range(n):
        await asyncio.sleep(0.01)
        yield i

evens = [x async for x in numbers(10) if x % 2 == 0]
print(evens)

# async generator expression (lazy version)
gen = (x * 2 async for x in numbers(5))
doubled = [x async for x in gen]
print(doubled)

### 22. Async Generator with Exception Handling

Write an async generator that can encounter an error mid-stream, handling it gracefully so the consumer gets partial results instead of crashing entirely.

In [ ]:
import asyncio

async def fetch_records(ids):
    for i in ids:
        await asyncio.sleep(0.01)
        if i == 3:
            print(f"  (skipping bad record {i})")
            continue              # skip this one, don't crash the whole stream
        yield f"record-{i}"

results = [r async for r in fetch_records(range(6))]
print(results)

### 23. Rate-Limited Batch Processor (real pattern)

Build a reusable `process_with_rate_limit(items, fn, max_concurrent)` helper combining `Semaphore` + `gather` — a pattern you'll reuse constantly for API integrations.

In [ ]:
import asyncio

async def process_with_rate_limit(items, fn, max_concurrent=3):
    sem = asyncio.Semaphore(max_concurrent)
    async def bounded(item):
        async with sem:
            return await fn(item)
    return await asyncio.gather(*[bounded(item) for item in items])

async def call_external_api(item):
    await asyncio.sleep(0.03)
    return f"processed-{item}"

results = await process_with_rate_limit(range(10), call_external_api, max_concurrent=3)
print(results)

### 24. Fan-Out / Fan-In with Result Aggregation

Build a fan-out/fan-in pattern: dispatch N sub-tasks (fan-out), then aggregate/reduce their results into one final value (fan-in).

In [ ]:
import asyncio

async def fetch_partial_sum(start, end):
    await asyncio.sleep(0.01)
    return sum(range(start, end))

async def parallel_sum(total, num_workers=4):
    chunk_size = total // num_workers
    ranges = [(i*chunk_size, (i+1)*chunk_size) for i in range(num_workers)]
    ranges[-1] = (ranges[-1][0], total)     # last chunk picks up any remainder

    partial_sums = await asyncio.gather(*[fetch_partial_sum(s, e) for s, e in ranges])
    return sum(partial_sums)    # fan-in: combine all partial results

result = await parallel_sum(1000, num_workers=4)
print(f"Parallel sum: {result}, expected: {sum(range(1000))}")

### 25. Async Context Manager (__aenter__/__aexit__)

Build a class-based ASYNC context manager `AsyncResource` (using `__aenter__`/`__aexit__`) simulating an async DB connection lifecycle.

In [ ]:
import asyncio

class AsyncResource:
    async def __aenter__(self):
        print("Acquiring resource (async)...")
        await asyncio.sleep(0.02)
        return self
    async def query(self, sql):
        await asyncio.sleep(0.01)
        return f"results for: {sql}"
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("Releasing resource (async)...")
        await asyncio.sleep(0.01)
        return False

async def main():
    async with AsyncResource() as res:
        result = await res.query("SELECT * FROM users")
        print(result)

await main()